# Job 1

Get atomic model and structure factor from REFMAC output

1. load the REFMAC reflections file
2. load the DIALS intensities file (Imerge)
3. figure out what ADP scaling was applied by REFMAC's Fobs
4. undo the ADP scaling to F columns
5. read in the PDB file and undo the ADP scaling of anisou parameters
6. save everything: Atoms, AtomFF, Basis, SpaceGroup, hklTable

In [1]:
% set up directories
mkdir proc
mkdir export

clear opts

% input files
opts.mtzREFMAC =   'import/hklout.mtz';
opts.mtzMerged =   'import/merged.mtz';
opts.pdbFileName = 'import/XYZOUT.pdb';

In [2]:
%%file anisoscaling.m

function [params,anisoscales] = anisoscaling(T,Basis,SpaceGroup)

SM = proc.script.ScaleModelToFobs('Basis',Basis.orient(),'SpaceGroup',SpaceGroup);

[T.sx,T.sy,T.sz] = SM.Basis.invert.frac2lab(T.h,T.k,T.l);

tt = T;
tt = tt(~isnan(tt.Imerge) & ~isnan(tt.Fobs) & tt.sigmaImerge>0,:); % get rid of NaNs

[numParams,param2struct] = SM.parameterize_model();
p2s = @(v) param2struct([v(1),0,0,v(2:(numParams-2))]); % HACK to remove ksol, Bsol from fitting model

Taniso = @(s) latt.Blob(1,0).addU(s.U).rescale(s.ktot);
residfun = @(s,t) (t.Imerge - abs(Taniso(s).scatteringAmplitude(t.sx,t.sy,t.sz).*t.Fobs).^2)./t.sigmaImerge;

solvFit = lsqnonlin(@(v) residfun(p2s(v/1000),tt),1000*[1,0,0],[],[]);

params = p2s(solvFit/1000);

anisoscales = latt.Blob(1,0).addU(params.U).scatteringAmplitude(T.sx,T.sy,T.sz);

end

Created file '/nfs/chess/user/spm82/mdx/mac1_20251105_goodvibes/anisoscaling.m'.


## 1. Load the REFMAC reflections file

In [3]:
opts.refmac_filter = struct('H','h','K','k','L','l','FREER','isFree','FP','Fobs','SIGFP','sigmaFobs',...
    'FC_ALL_LS','Fcalc','PHIC_ALL_LS','phiFcalc','FC','Fatom','PHIC','phiFatom');

% load refmac reflections file
[hklREFMAC,Basis,SpaceGroup] = proc.script.ImportMTZ('mtz2mdx',opts.refmac_filter).run(opts.mtzREFMAC);

% reformat the table (hklREFMAC --> T1)
T1 = hklREFMAC(:,{'h','k','l','isFree','Fobs','sigmaFobs','Fcalc','Fatom'});
T1.Fcalc = T1.Fcalc.*exp(1i*(pi/180).*hklREFMAC.phiFcalc); % add phases
T1.Fatom = T1.Fatom.*exp(1i*(pi/180).*hklREFMAC.phiFatom); % add phases

In [4]:
head(T1)

    h    k    l     isFree     Fobs     sigmaFobs         Fcalc              Fatom     
    _    _    __    ______    ______    _________    _______________    _______________

    0    0     4       5      956.57     14.179       1469.3+209.29i     1152.1+352.78i
    0    0     8      27      875.07     10.524       -457.1+1089.1i    -675.53+1180.7i
    0    0    12       7      856.43     10.327       664.22-240.87i     679.01-151.37i
    0    0    16       9      723.67     8.7592       585.34+189.99i     563.13+193.32i
    0    0    20      60      20.597     3.4482      -3.8074-7.6537i    -2.5861-5.3833i
    0    0    24      51       158.8     2.1723       137.94+86.292i     138.07+86.445i
    0    0    28      12       55.92     1.5195      -38.602-49.774i    -38.346-49.826i
    0    0    32      42      69.843     2.3576      -25.454-63.958i    -25.447-63.932i



## 2. Load the DIALS intensities file (Imerge)

In [5]:
opts.merged_filter = struct('H','h','K','k','L','l','IMEAN','Imerge','SIGIMEAN','sigmaImerge');
hklMerged = proc.script.ImportMTZ('mtz2mdx',opts.merged_filter).run(opts.mtzMerged); % Imerge

% reformat the table (hklMerged --> T2)
T2 = hklMerged(:,{'h','k','l','Imerge','sigmaImerge'});
T2 = T2(~isnan(T2.Imerge),:);
head(T2)

    h    k    l    Imerge     sigmaImerge
    _    _    _    _______    ___________

    0    0    1    0.14662     0.028269  
    0    0    2    0.20819      0.03488  
    0    0    3    0.19341      0.04674  
    0    0    4     5597.4       156.42  
    0    0    5    0.50592      0.10305  
    0    0    6    0.20028      0.10505  
    0    0    7    0.28511      0.12319  
    0    0    8     4592.2        110.4  



## 3. Figure out what ADP scaling was applied by REFMAC's Fobs

In [6]:
% combine tables (T <-- T1, T2)
T = outerjoin(T1,T2,'Keys',{'h','k','l'},'MergeKeys',true);

[params,anisoscales] = anisoscaling(T,Basis,SpaceGroup);
Uadd = params.U;
disp(Uadd)


Local minimum possible.

lsqnonlin stopped because the final change in the sum of squares relative to 
its initial value is less than the value of the function tolerance.

   -0.0088         0         0
         0   -0.0088         0
         0         0    0.0170



## 4. Undo the ADP scaling to F columns

In [7]:
hklTable = T;
hklTable.Fobs = hklTable.Fobs.*anisoscales;
hklTable.sigmaFobs = hklTable.sigmaFobs.*anisoscales;
hklTable.Fcalc = hklTable.Fcalc.*anisoscales;
hklTable.Fatom = hklTable.Fatom.*anisoscales;

## 5. Read in the PDB file and undo the ADP scaling of anisou parameters

In [8]:
Atoms = proc.script.ImportPDB().run(opts.pdbFileName);
AtomFF = proc.script.CoordinateTools.to_xray_structure(Atoms);

% apply anisotropic ADP to atomic form factors
AtomFF.mdxFormFactor = AtomFF.mdxFormFactor.addU(Uadd);

## 6. Save everything: Atoms, AtomFF, Basis, SpaceGroup, hklTable

In [10]:
save proc/atomic_model.mat Atoms AtomFF Basis SpaceGroup hklTable opts Uadd